# 🕵️ Catch the Fraudster: Part A

You work at a payments company. Millions of transactions flow through. A tiny fraction are fraud.
**Your job: build models, and learn when they can be trusted.**

**How this notebook works:** you never write code. Each cell below is a control panel (dropdowns, sliders, checkboxes).
Change the controls, press ▶ on the cell, read the result. Run the **Setup** cell first.

> Before every run, say out loud (or write down): *"I think the score will go UP / DOWN because..."*

In [ ]:
#@title Setup: check the data link, then press ▶ { display-mode: "form" }
DATA_URL = "https://github.com/DakshTomar0223/transac/blob/main/workshop_part_a.ipynb" #@param {type:"string"}

import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

CATEGORICAL = ["transaction_type", "merchant_category", "location", "device_used", "payment_channel"]
BASIC = ["amount", "hour", "day_of_week"] + CATEGORICAL
BEHAVIOR = ["time_since_last_transaction", "spending_deviation_score", "velocity_score", "geo_anomaly_score"]
NOT_FRAUD = {"", "none", "nan", "null", "no_fraud", "not_fraud", "legit", "legitimate", "normal", "n/a", "na"}
NEEDED = ["timestamp", "amount", "transaction_type", "merchant_category", "location", "device_used",
          "payment_channel", "time_since_last_transaction", "spending_deviation_score", "velocity_score",
          "geo_anomaly_score", "fraud_type", "is_fraud"]


def load_data():
    url = DATA_URL.strip()
    if url and not url.startswith("PASTE"):
        return pd.read_csv(url, parse_dates=["timestamp"])
    for p in ["sample.csv", "sample.csv.gz", "data/sample.csv.gz", "data/sample.csv"]:
        if os.path.exists(p):
            return pd.read_csv(p, parse_dates=["timestamp"])
    try:
        from google.colab import files
        print("No link given. Upload sample.csv now...")
        name = list(files.upload().keys())[0]
        return pd.read_csv(name, parse_dates=["timestamp"])
    except ImportError:
        raise FileNotFoundError("Data not found. Paste the link in DATA_URL or put sample.csv next to this notebook.")


df = load_data()
miss = [c for c in NEEDED if c not in df.columns]
if miss:
    raise ValueError(f"These columns are missing from the data: {miss}")
df = df.sort_values("timestamp").reset_index(drop=True)
df["hour"] = df.timestamp.dt.hour
df["day_of_week"] = df.timestamp.dt.dayofweek
ft = df["fraud_type"].fillna("").astype(str).str.strip().str.lower()
df["fraud_type_known"] = (~ft.isin(NOT_FRAUD)).astype(int)
QUIRK = df[["time_since_last_transaction", "is_fraud"]].copy()

n = len(df)
TRAIN, VAL, TEST = (df.iloc[: int(n * .65)].copy(), df.iloc[int(n * .65): int(n * .80)].copy(), df.iloc[int(n * .80):].copy())
for c in BEHAVIOR + ["amount"]:
    med = TRAIN[c].median()
    for part in (TRAIN, VAL, TEST):
        part[c] = part[c].fillna(med)
CAT_LEVELS = {}
for c in CATEGORICAL:
    top = TRAIN[c].fillna("unknown").value_counts().index[:12].tolist()
    CAT_LEVELS[c] = top + ["other"]
    for part in (TRAIN, VAL, TEST):
        v = part[c].fillna("unknown")
        part[c] = v.where(v.isin(top), "other")
TEST["fraud_type_known"] = 0
TRAIN, VAL, TEST = TRAIN.reset_index(drop=True), VAL.reset_index(drop=True), TEST.reset_index(drop=True)


def make_X(d, feats):
    cols = {}
    for f in feats:
        if f in CATEGORICAL:
            for lvl in CAT_LEVELS[f]:
                cols[f"{f}={lvl}"] = (d[f] == lvl).astype(np.int8).values
        else:
            cols[f] = d[f].values.astype(float)
    return pd.DataFrame(cols, index=d.index)


class Wrapper:
    def __init__(self, algo, complexity, feats):
        self.algo, self.complexity, self.feats, self.model, self.thr = algo, complexity, list(feats), None, {}

    def fit(self, d):
        if self.algo == "Never fraud (lazy)":
            return self
        if self.algo == "Simple rules":
            self.thr = {c: float(np.quantile(np.abs(d[c]), 0.95)) for c in BEHAVIOR}
            return self
        X, y, k = make_X(d, self.feats), d.is_fraud, int(self.complexity)
        if self.algo == "Logistic regression":
            m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
        elif self.algo == "Decision tree":
            m = DecisionTreeClassifier(max_depth=k, random_state=0)
        elif self.algo == "Random forest":
            m = RandomForestClassifier(n_estimators=30, max_depth=k, random_state=0, n_jobs=-1)
        elif self.algo == "Boosted trees":
            m = HistGradientBoostingClassifier(max_depth=k, max_iter=100, random_state=0)
        self.model = m.fit(X, y)
        return self

    def proba(self, d):
        if self.algo == "Never fraud (lazy)":
            return np.zeros(len(d))
        if self.algo == "Simple rules":
            hit = np.zeros(len(d), dtype=bool)
            for c, t in self.thr.items():
                hit |= np.abs(d[c].values) > t
            return hit.astype(float)
        return self.model.predict_proba(make_X(d, self.feats))[:, 1]


def metrics(y, pred):
    return dict(Accuracy=accuracy_score(y, pred), Precision=precision_score(y, pred, zero_division=0),
                Recall=recall_score(y, pred, zero_division=0), F1=f1_score(y, pred, zero_division=0))


LOG = []


def plot_log():
    t = pd.DataFrame([{k: e[k] for k in ("id", "train_F1", "val_F1")} for e in LOG])
    fig, ax = plt.subplots(figsize=(max(5, 0.7 * len(t) + 2), 3))
    x, w = np.arange(len(t)), 0.38
    ax.bar(x - w / 2, t.train_F1, w, label="Training F1 (what it memorised)")
    ax.bar(x + w / 2, t.val_F1, w, label="Validation F1 (what it really knows)")
    ax.set_xticks(x); ax.set_xticklabels([f"#{i}" for i in t.id]); ax.set_xlabel("experiment"); ax.set_ylabel("F1")
    ax.legend(fontsize=8); plt.tight_layout(); plt.show()


def run_experiment(algo, complexity, threshold, feats, log=True, verbose=True, plot=True, note=""):
    m = Wrapper(algo, complexity, feats).fit(TRAIN)
    p_tr, p_va = m.proba(TRAIN), m.proba(VAL)
    mt, mv = metrics(TRAIN.is_fraud, p_tr >= threshold), metrics(VAL.is_fraud, p_va >= threshold)
    e = dict(id=len(LOG), algo=algo, complexity=int(complexity), threshold=threshold, clues=len(feats),
             mystery_col="fraud_type_known" in feats, train_F1=round(mt["F1"], 3), val_F1=round(mv["F1"], 3),
             gap=round(mt["F1"] - mv["F1"], 3), val_proba=p_va.astype(np.float32),
             prod_proba=m.proba(TEST).astype(np.float32), note=note)
    if log:
        LOG.append(e)
    if verbose:
        print(f"{algo} | complexity {int(complexity)} | threshold {threshold} | clues used: {len(feats)}")
        display(pd.DataFrame({"Training": mt, "Validation": mv}).round(3))
        print(f"Training F1 minus Validation F1 = {e['gap']:+.2f}" + ("   <-- look at that gap" if e["gap"] > 0.25 else ""))
        if mv["F1"] == 0 and mv["Recall"] == 0:
            print("(F1 = 0 means the model flagged NO fraud at all. Try lowering the threshold.)")
        if log and plot:
            plot_log()
    return e


def log_table():
    t = pd.DataFrame([{k: v for k, v in e.items() if k not in ("val_proba", "prod_proba")} for e in LOG])
    display(t.set_index("id"))


print(f"Ready. {len(TRAIN):,} training rows | {len(VAL):,} validation rows | fraud rate {df.is_fraud.mean():.2%}")

## Round 0: Spot the fraud 🔍

Look at these 10 real transactions. **Vote: which ones are fraud?** Write your guesses first, then tick the box.

In [ ]:
#@title Round 0: Spot the fraud { display-mode: "form" }
SHOW_ANSWERS = False #@param {type:"boolean"}

s = pd.concat([VAL[VAL.is_fraud == 1].sample(3, random_state=5), VAL[VAL.is_fraud == 0].sample(7, random_state=5)]).sample(frac=1, random_state=9)
cols = ["timestamp", "amount", "transaction_type", "merchant_category", "location", "device_used", "payment_channel",
        "time_since_last_transaction", "spending_deviation_score", "velocity_score", "geo_anomaly_score"]
v = s[cols].reset_index(drop=True)
v.index = v.index + 1
if SHOW_ANSWERS:
    v["FRAUD?"] = s.is_fraud.map({1: "YES", 0: "no"}).values
display(v)
print(f"Real life: only about {df.is_fraud.mean():.1%} of transactions are fraud. 3 in 10 is unrealistically high.")

## Round 1: The accuracy trap 🪤

First, a model that **never** flags fraud. Then a real model. Slide the threshold and run again.
Lower threshold = flag more transactions. Higher threshold = flag fewer.

In [ ]:
#@title Round 1: The accuracy trap { display-mode: "form" }
THRESHOLD = 0.5 #@param {type:"slider", min:0.05, max:0.95, step:0.05}

lazy = Wrapper("Never fraud (lazy)", 1, BASIC + BEHAVIOR).fit(TRAIN)
lm = metrics(VAL.is_fraud, lazy.proba(VAL) >= 0.5)
print("--- LAZY MODEL: always says 'not fraud' ---")
print(f"Accuracy: {lm['Accuracy']:.1%}     Fraud caught (recall): {lm['Recall']:.1%}\n")
print("--- REAL MODEL: logistic regression ---")
e = run_experiment("Logistic regression", 1, THRESHOLD, BASIC + BEHAVIOR, plot=False, note="round 1")
pred, actual = e["val_proba"] >= THRESHOLD, VAL.is_fraud.values
display(pd.DataFrame({"flagged legit": [((~pred) & (actual == 0)).sum(), ((~pred) & (actual == 1)).sum()],
                      "flagged fraud": [(pred & (actual == 0)).sum(), (pred & (actual == 1)).sum()]},
                     index=["actually legit", "actually fraud"]))

**Discuss:** The lazy model is ~99% accurate and has caught zero fraud. Why is accuracy a terrible score here?
What do **precision** (of the ones I flagged, how many were fraud?) and **recall** (of all fraud, how much did I catch?) tell you?

## Round 2: Algorithm face-off, overfitting and the mystery column

Pick an algorithm, set **complexity** (how many levels of questions a tree may ask), set the **threshold**.
Compare **Training** vs **Validation**. Then try the same algorithm at complexity 3, 10, then 30.

**Try this:** tick `USE_MYSTERY_COLUMN` and run again. Is the score believable?

In [ ]:
#@title Round 2: Algorithm face-off { display-mode: "form" }
ALGORITHM = "Decision tree" #@param ["Simple rules", "Logistic regression", "Decision tree", "Random forest", "Boosted trees"]
COMPLEXITY = 6 #@param {type:"slider", min:1, max:30, step:1}
THRESHOLD = 0.5 #@param {type:"slider", min:0.05, max:0.95, step:0.05}
USE_BEHAVIOUR_SCORES = True #@param {type:"boolean"}
USE_MYSTERY_COLUMN = False #@param {type:"boolean"}

feats = list(BASIC) + (list(BEHAVIOR) if USE_BEHAVIOUR_SCORES else []) + (["fraud_type_known"] if USE_MYSTERY_COLUMN else [])
run_experiment(ALGORITHM, COMPLEXITY, THRESHOLD, feats, note="round 2")
print("\nEXPERIMENT LOG")
log_table()

## Round 2b: Deploy to production 🚀

Your best model on validation data looked great. Now deploy it. **In real life the mystery column does not exist yet**
when a transaction arrives, because investigators only fill it in after fraud is confirmed.

In [ ]:
#@title Round 2b: Deploy to production { display-mode: "form" }
EXPERIMENT_NUMBER = -1 #@param {type:"integer"}

if not LOG:
    raise ValueError("Run Round 2 first.")
e = LOG[-1] if EXPERIMENT_NUMBER < 0 else LOG[EXPERIMENT_NUMBER]
val_m = metrics(VAL.is_fraud, e["val_proba"] >= e["threshold"])
prod_m = metrics(TEST.is_fraud, e["prod_proba"] >= e["threshold"])
print(f"Experiment #{e['id']}: {e['algo']} (complexity {e['complexity']}), used the mystery column: {e['mystery_col']}\n")
display(pd.DataFrame({"On validation data (testing)": val_m, "In production (real life)": prod_m}).round(3))

**Discuss:** A model with a "too good to be true" score usually is. What is **data leakage**? How could the skeptic in your team have caught it?
And in Round 2: if a deep tree scores 100% on training but ~0% on validation, what did it actually learn?

> Honest result on THIS data: no honest model does much better than guessing. **If the data has no signal, no algorithm can create it.**
> In Part B you will get data that *does* have a pattern worth finding.

## Instructor only (delete before sharing)

In [ ]:
#@title Bonus (instructor): the strange column { display-mode: "form" }
SHOW = False #@param {type:"boolean"}

if SHOW:
    q = QUIRK.copy()
    q["time_since_last_transaction is"] = np.where(q.time_since_last_transaction.isna(), "missing", "filled in")
    t = q.groupby("time_since_last_transaction is").is_fraud.agg(["size", "sum", "mean"])
    t.columns = ["rows", "fraud rows", "fraud rate"]
    display(t.round(4))
    print("Whether a field is empty can itself give the answer away. Another form of leakage.")